# 🧪 Prompt Engineering for LLMs (Built on Attention Theory)

## What You'll Learn
- Why **position** matters in prompts (the "Lost in the Middle" effect)
- How **context length** dilutes attention to important tokens
- How **semantic similarity** drives attention weight — and why RAG works
- A practical **prompt-engineering cheat sheet** grounded in attention math

## Prerequisites
This notebook assumes you have worked through `06_Attention_Mechanism.ipynb`.
We use the `scaled_dot_product_attention` function you implemented there to
run experiments that show *how* attention behaves in realistic prompt setups.

---

In [ ]:
# Setup — import ``scaled_dot_product_attention`` from the shared utils
# so this notebook does not depend on definitions in notebook 06.
import sys
import numpy as np
import mlx.core as mx
import matplotlib.pyplot as plt

from mlx_nlp_utils import (
    print_device_info,
    scaled_dot_product_attention,
)

print_device_info()


---

## Next Steps

Now that you understand:
1. **The math** (Notebook 06 core): How attention works
2. **The behavior** (this notebook): How attention affects prompt engineering

You're ready to build the **Transformer** architecture in the next notebook: **[07_Build_NanoGPT.ipynb](07_Build_NanoGPT.ipynb)**.

### 📋 Prompt Engineering Cheat Sheet (Based on Attention)

| Attention Insight | Prompt Engineering Rule |
|-------------------|------------------------|
| Softmax creates competition | Don't overload context - tokens compete for attention |
| Position bias exists | Put critical info at START or END, not middle |
| Similar embeddings → high attention | Use RAG to retrieve relevant context |
| Causal mask limits visibility | For chat, recent messages are most "visible" |
| Longer context = diluted attention | Summarize long histories, don't dump everything |

### 🧪 Try These Experiments with Real Models

Once you complete Notebook 08 (LoRA Fine-Tuning), come back and try:

```python
# Test position effects with a real model
from mlx_lm import load, generate

model, tokenizer = load("mlx-community/Llama-3.2-1B-Instruct-4bit")

# Variant A: Instruction at beginning
prompt_A = "IMPORTANT: Always mention the word APPLE. Tell me about fruits."

# Variant B: Instruction at end  
prompt_B = "Tell me about fruits. IMPORTANT: Always mention the word APPLE."

# Variant C: Instruction in middle (buried)
prompt_C = "Tell me about fruits. [filler text here] IMPORTANT: Always mention APPLE. [more filler] Give details."

# Compare: Which variant follows the instruction best?
```

In [ ]:
# Simulating semantic similarity in attention
# When query is similar to a key, attention should be higher

d_model_sim = 8

# Create a "query" vector
mx.random.seed(42)
query_vec = mx.random.normal((1, 1, d_model_sim))

# Create keys with varying similarity to the query
# Key 0: Very similar (query + small noise)
# Key 1-3: Random (unrelated)
# Key 4: Somewhat similar (query + medium noise)

similar_key = query_vec + mx.random.normal((1, 1, d_model_sim)) * 0.1  # Very similar
somewhat_similar = query_vec + mx.random.normal((1, 1, d_model_sim)) * 0.5  # Somewhat
random_keys = mx.random.normal((1, 3, d_model_sim))  # Unrelated

# Stack: [very_similar, random, random, random, somewhat_similar]
K_semantic = mx.concatenate([similar_key, random_keys, somewhat_similar], axis=1)
V_semantic = mx.random.normal((1, 5, d_model_sim))

_, weights_semantic = scaled_dot_product_attention(query_vec, K_semantic, V_semantic)

labels = ["Very Similar", "Random 1", "Random 2", "Random 3", "Somewhat Similar"]
weights_sem_np = np.array(weights_semantic[0, 0])

plt.figure(figsize=(10, 5))
colors = ['green', 'gray', 'gray', 'gray', 'lightgreen']
bars = plt.bar(labels, weights_sem_np, color=colors)
plt.ylabel("Attention Weight")
plt.title("Semantic Similarity → Stronger Attention\n(Query attends more to similar keys)")
plt.xticks(rotation=15)

for bar, weight in zip(bars, weights_sem_np):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{weight:.3f}', ha='center', va='bottom')

plt.show()

print("💡 This is why RAG works!")
print("   - Retrieving semantically similar context → stronger attention")
print("   - Irrelevant context gets ignored (low attention)")
print("   - Quality of retrieval directly impacts generation quality")

### Experiment 3: Semantic Similarity Effects

**Research Question:** Do semantically similar tokens attend to each other more strongly?

This is the foundation of why RAG (Retrieval Augmented Generation) works - retrieving semantically similar context helps because similar embeddings create stronger attention patterns.

In [ ]:
# How does attention to the FIRST token change as context grows?

context_lengths = [5, 10, 20, 50, 100]
attention_to_first = []

mx.random.seed(42)

for ctx_len in context_lengths:
    Q_test = mx.random.normal((1, 1, 8))
    K_test = mx.random.normal((1, ctx_len, 8))
    V_test = mx.random.normal((1, ctx_len, 8))
    
    _, weights_test = scaled_dot_product_attention(Q_test, K_test, V_test)
    
    # How much attention goes to the first token?
    first_token_attention = float(weights_test[0, 0, 0])
    attention_to_first.append(first_token_attention)

# Plot degradation
plt.figure(figsize=(10, 5))
plt.plot(context_lengths, attention_to_first, 'bo-', linewidth=2, markersize=10)
plt.xlabel("Context Length (number of tokens)")
plt.ylabel("Attention to First Token")
plt.title("Attention Degradation: How 'Visible' is Position 0?")
plt.grid(True, alpha=0.3)

# Add annotations
for i, (x, y) in enumerate(zip(context_lengths, attention_to_first)):
    plt.annotate(f'{y:.3f}', (x, y), textcoords="offset points", xytext=(0,10), ha='center')

plt.show()

print("💡 Key Insight:")
print("   - With 5 tokens, first token gets ~20% attention (1/5 = 0.20)")
print("   - With 100 tokens, first token might get <1% attention")
print("   - This is why summarization/chunking matters for long contexts!")

### Experiment 2: Context Length Degradation

**Research Question:** How does attention quality degrade as context gets longer?

As context grows, the softmax must distribute attention across more tokens, potentially diluting focus on important information.

In [ ]:
# Simulating "Lost in the Middle" with Attention
# Let's see how attention naturally decays with position

seq_len_long = 20  # Simulate a longer context

mx.random.seed(42)
Q_long = mx.random.normal((1, 1, 8))  # Single query (like the final generation token)
K_long = mx.random.normal((1, seq_len_long, 8))  # 20 context tokens
V_long = mx.random.normal((1, seq_len_long, 8))

# Compute attention from final position to all previous positions
output_long, weights_long = scaled_dot_product_attention(Q_long, K_long, V_long)

weights_long_np = np.array(weights_long[0, 0])  # Shape: (20,)

# Visualize attention distribution
plt.figure(figsize=(12, 4))
positions = list(range(seq_len_long))
colors = ['green' if i < 3 else 'red' if i > 16 else 'orange' for i in positions]

plt.bar(positions, weights_long_np, color=colors)
plt.xlabel("Token Position in Context")
plt.ylabel("Attention Weight")
plt.title("Attention Distribution Across Context Length\n(Green=Beginning, Orange=Middle, Red=End)")
plt.axhline(y=1/seq_len_long, color='black', linestyle='--', label='Uniform attention')
plt.legend()
plt.show()

print("💡 Implications for Prompt Engineering:")
print("   - Put critical instructions at the BEGINNING or END")
print("   - Don't bury important context in the middle")
print("   - If using few-shot examples, put the best ones first or last")

### Experiment 1: Position Effects

**Research Question:** Does it matter WHERE you put important information in a prompt?

**The "Lost in the Middle" Phenomenon:** Research (Liu et al., 2023) shows that LLMs tend to:
- Pay strong attention to the **beginning** of prompts
- Pay strong attention to the **end** of prompts  
- Often **ignore information in the middle**

This has huge implications for prompt design!

## 📝 Prompt Engineering Cheat Sheet (Based on Attention Math)

Now that you understand the math, here is how to write better prompts:

| Attention Property | Prompting Rule | Why? |
|-------------------|----------------|------|
| **Positional Bias** | Put instructions at **start/end** | Attention sinks/anchors are strongest at boundaries. |
| **Self-Attention** | Use **consistent terminology** | $QK^T$ looks for similarity. "Car" matches "Vehicle" less than "Car" matches "Car". |
| **Context Window** | **Remove fluff** | Every token competes for attention probability (Softmax sums to 1). Noise dilutes signal. |
| **Causal Masking** | **Chain of Thought** works | The model can only attend to *past* tokens. It needs to "think" (generate tokens) before it can "solve". |